In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.lines import Line2D
from pathlib import Path

# --------------------------
# 1. Config
# --------------------------
base_dir = Path("/Users/ewellmeyer/Documents/research/weights")

# Select which channel counts to include
channels = [2, 4, 8, 16, 32, 64, 128]

# gn_groups used per channel during training (all 1 by default for unet6R runs)
gn_per_ch = {ch: 1 for ch in channels}

# Fixed training parameters (must match what train_pr_dpdk.py used)
k_size    = 3
num_bins  = 64
dP_min    = -700
dP_max    = 1200

def ens_dir(ch):
    gn = gn_per_ch[ch]
    name = (
        f"unet_ens_HG789_PR_dPdK_Softmax_unet6R_ch{ch}_k{k_size}_"
        f"128x_dPbins{num_bins}_gn{gn}_dpmin{dP_min}_dPmax{dP_max}"
    )
    return base_dir / name

# --------------------------
# 2. Load results
# --------------------------
softmax_rmse = []
baseline_rmse = None

for ch in channels:
    npz_path = ens_dir(ch) / "softmax_ensemble_analysis_arrays.npz"
    try:
        d = np.load(npz_path)
        idx = d['test_indices']
        softmax_rmse.append(np.median(d['rmse_softmax_mean'][idx]))
        if baseline_rmse is None:
            baseline_rmse = np.median(d['rmse_ppe'][idx])
    except FileNotFoundError:
        softmax_rmse.append(np.nan)
        print(f"Missing results for ch={ch}: {npz_path}")

# --------------------------
# 3. Plot
# --------------------------
sns.set_context("paper", font_scale=1.4)
sns.set_style("ticks")

fig, ax = plt.subplots(figsize=(8, 6))

c_soft = '#1f77b4'

if baseline_rmse is not None:
    ax.axhline(baseline_rmse, color='gray', linestyle=':', linewidth=1.5, alpha=0.8, zorder=1)
    ax.text(channels[0], baseline_rmse, 'PPE mean - PPE Test', color='gray',
            fontsize=9, va='bottom', ha='left', fontweight='bold')

ax.plot(channels, softmax_rmse, color=c_soft, linestyle='-', marker='o', lw=2, ms=6)

ax.set_xlabel("Network Channels per Layer", fontsize=12)
ax.set_ylabel(r"Median RMSE (mm yr$^{-1}$ K$^{-1}$)", fontsize=12, fontweight='bold')
ax.set_xscale('log', base=2)
ax.set_xticks(channels)
ax.set_xticklabels(channels)
ax.set_ylim(35, 90)
ax.set_xlim(1.5, 135)
ax.grid(True, which='major', linestyle='--', alpha=0.4)
sns.despine(ax=ax)

legend_elements = [
    Line2D([0], [0], color=c_soft, lw=3, marker='o', ms=6, label='Base Model (HadGEM Test)'),
    Line2D([0], [0], color='gray', linestyle=':', lw=2, label='HadGEM Baseline (PPE Train mean)'),
]
ax.legend(handles=legend_elements, frameon=False, fontsize=11)

plt.tight_layout()
plt.show()